# Skin Disease Classification
Standalone Kaggle notebook - all code inlined for one-click execution


In [ ]:
!python --version

## Setup dependencies

In [ ]:
!apt-get update
!apt-get install -y wget ffmpeg libsm6 libxext6
!pip install torch==2.6.0+cu126 torchvision==0.21.0+cu126 torchaudio==2.6.0+cu126 --index-url https://download.pytorch.org/whl/cu126
!pip install numpy==2.2.2 matplotlib==3.10.3 seaborn==0.13.2 scikit-learn==1.7.1 Pillow==11.1.0 transformers==4.50.0 timm==1.0.9 einops==0.8.1 grad-cam==1.5.5 requests==2.32.3

In [ ]:
!wget https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4+cu12torch2.6cxx11abiTRUE-cp311-cp311-linux_x86_64.whl
!pip install mamba_ssm-2.2.4+cu12torch2.6cxx11abiTRUE-cp311-cp311-linux_x86_64.whl
!rm mamba_ssm-2.2.4+cu12torch2.6cxx11abiTRUE-cp311-cp311-linux_x86_64.whl

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import json
import shutil
from collections import OrderedDict
from contextlib import suppress
from datetime import datetime
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from timm.data import create_dataset, create_loader, resolve_data_config
from timm.models import model_parameters
from timm import utils
from timm.loss import SoftTargetCrossEntropy, LabelSmoothingCrossEntropy
from timm.optim import create_optimizer_v2, optimizer_kwargs
from timm.scheduler import create_scheduler
from timm.utils import NativeScaler
from timm.models.layers import trunc_normal_, DropPath, LayerNorm2d
from timm.models.vision_transformer import Mlp

from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
from einops import rearrange, repeat

torch.backends.cudnn.benchmark = True


## Model's main code

In [ ]:
def window_partition(x, window_size):
    """Partition feature map into windows"""
    B, C, H, W = x.shape
    x = x.view(B, C, H // window_size, window_size, W // window_size, window_size)
    windows = x.permute(0, 2, 4, 3, 5, 1).reshape(-1, window_size*window_size, C)
    return windows

def window_reverse(windows, window_size, H, W):
    """Reverse window partitioning"""
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.reshape(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 5, 1, 3, 2, 4).reshape(B, windows.shape[2], H, W)
    return x

class EfficientDownsample(nn.Module):
    """Lightweight downsampling with depthwise separable convolution"""
    def __init__(self, dim, keep_dim=False):
        super().__init__()
        dim_out = dim if keep_dim else 2 * dim
        self.reduction = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 2, 1, groups=dim, bias=False),
            nn.Conv2d(dim, dim_out, 1, 1, 0, bias=False),
            nn.BatchNorm2d(dim_out, eps=1e-4)
        )
    def forward(self, x):
        return self.reduction(x)

class LightweightPatchEmbed(nn.Module):
    """Efficient patch embedding with reduced parameters"""
    def __init__(self, in_chans=3, in_dim=16, dim=48):
        super().__init__()
        self.conv_down = nn.Sequential(
            nn.Conv2d(in_chans, in_dim, 3, 2, 1, bias=False),
            nn.BatchNorm2d(in_dim, eps=1e-4),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_dim, dim, 3, 2, 1, bias=False),
            nn.BatchNorm2d(dim, eps=1e-4),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv_down(x)

class DiseaseAwareMixer(nn.Module):
    """Lightweight Mamba mixer optimized for disease pattern recognition"""
    def __init__(self, d_model, d_state=8, d_conv=3, expand=1.5):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = int(self.expand * self.d_model)
        self.dt_rank = max(math.ceil(self.d_model / 16), 1)
        self.in_proj = nn.Linear(self.d_model, self.d_inner, bias=False)
        self.x_proj = nn.Linear(self.d_inner//2, self.dt_rank + self.d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner//2, bias=True)
        dt_init_std = self.dt_rank**-0.5
        nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
        dt = torch.exp(torch.rand(self.d_inner//2) * (math.log(0.1) - math.log(0.001)) + math.log(0.001))
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_dt)
        self.dt_proj.bias._no_reinit = True
        A = repeat(torch.arange(1, self.d_state + 1, dtype=torch.float32), "n -> d n", d=self.d_inner//2)
        A_log = torch.log(A)
        self.A_log = nn.Parameter(A_log)
        self.A_log._no_weight_decay = True
        self.D = nn.Parameter(torch.ones(self.d_inner//2))
        self.D._no_weight_decay = True
        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=False)
        self.conv1d_x = nn.Conv1d(self.d_inner//2, self.d_inner//2, d_conv, 
                                 padding=(d_conv-1)//2, groups=self.d_inner//2, bias=False)
        self.conv1d_z = nn.Conv1d(self.d_inner//2, self.d_inner//2, d_conv, 
                                 padding=(d_conv-1)//2, groups=self.d_inner//2, bias=False)

    def forward(self, hidden_states):
        _, seqlen, _ = hidden_states.shape
        xz = self.in_proj(hidden_states)
        xz = rearrange(xz, "b l d -> b d l")
        x, z = xz.chunk(2, dim=1)
        A = -torch.exp(self.A_log.float())
        x = F.silu(self.conv1d_x(x))
        z = F.silu(self.conv1d_z(z))
        x_dbl = self.x_proj(rearrange(x, "b d l -> (b l) d"))
        dt, B, C = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = rearrange(self.dt_proj(dt), "(b l) d -> b d l", l=seqlen)
        B = rearrange(B, "(b l) dstate -> b dstate l", l=seqlen).contiguous()
        C = rearrange(C, "(b l) dstate -> b dstate l", l=seqlen).contiguous()
        y = selective_scan_fn(x, dt, A, B, C, self.D.float(), z=None, 
                             delta_bias=self.dt_proj.bias.float(), delta_softplus=True)
        y = torch.cat([y, z], dim=1)
        y = rearrange(y, "b d l -> b l d")
        return self.out_proj(y)

class MultiScaleMambaMixer(nn.Module):
    """MS2D-inspired multi-scale SSM mixer"""
    def __init__(self, d_model: int, d_state: int = 8, d_conv: int = 3, expand: float = 1.5):
        super().__init__()
        self.high = DiseaseAwareMixer(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.low = DiseaseAwareMixer(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=1.0)
        self.gate = nn.Parameter(torch.zeros(1, 1, d_model))

    def forward(self, x):
        B, L, C = x.shape
        S = int(L ** 0.5)
        y_high = self.high(x)
        x_img = x.transpose(1, 2).reshape(B, C, S, S)
        Sl = max(1, S // 2)
        x_low = F.interpolate(x_img, size=(Sl, Sl), mode='bilinear', align_corners=False)
        x_low_seq = x_low.flatten(2).transpose(1, 2)
        y_low = self.low(x_low_seq)
        y_low_img = y_low.transpose(1, 2).reshape(B, C, Sl, Sl)
        y_low_up = F.interpolate(y_low_img, size=(S, S), mode='bilinear', align_corners=False)
        y_low_up = y_low_up.flatten(2).transpose(1, 2)
        g = torch.sigmoid(self.gate)
        return (1.0 - g) * y_high + g * y_low_up

class StandardMLP(nn.Module):
    """Standard MLP with GELU activation"""
    def __init__(self, in_features: int, hidden_features: int, drop: float = 0.0):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, in_features)
        self.dropout = nn.Dropout(drop)
    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return self.dropout(x)

class SkinDiseaseMambaBlock(nn.Module):
    """Pure Mamba block with standard MLP"""
    def __init__(self, dim, mlp_ratio=2., drop=0., drop_path=0., layer_scale=None):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.mixer = MultiScaleMambaMixer(d_model=dim, d_state=8, d_conv=3, expand=1.5)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * (mlp_ratio * 0.8))
        self.mlp = StandardMLP(in_features=dim, hidden_features=mlp_hidden_dim, drop=drop)
        self.gamma_1 = nn.Parameter(layer_scale * torch.ones(dim)) if layer_scale else 1
        self.gamma_2 = nn.Parameter(layer_scale * torch.ones(dim)) if layer_scale else 1
    def forward(self, x):
        x = x + self.drop_path(self.gamma_1 * self.mixer(self.norm1(x)))
        x = x + self.drop_path(self.gamma_2 * self.mlp(self.norm2(x)))
        return x

class SkinDiseaseMambaStage(nn.Module):
    """Efficient stage with windowed processing"""
    def __init__(self, dim, depth, num_heads, window_size, downsample=True,
                 mlp_ratio=2., drop=0., attn_drop=0., drop_path=0., layer_scale=None):
        super().__init__()
        self.blocks = nn.ModuleList([
            SkinDiseaseMambaBlock(
                dim=dim, mlp_ratio=mlp_ratio, drop=drop,
                drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                layer_scale=layer_scale
            ) for i in range(depth)
        ])
        self.downsample = EfficientDownsample(dim=dim) if downsample else None
        self.window_size = window_size
    def forward(self, x):
        _, _, H, W = x.shape
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        if pad_r > 0 or pad_b > 0:
            x = F.pad(x, (0, pad_r, 0, pad_b))
            _, _, Hp, Wp = x.shape
        else:
            Hp, Wp = H, W
        x = window_partition(x, self.window_size)
        for blk in self.blocks:
            x = blk(x)
        x = window_reverse(x, self.window_size, Hp, Wp)
        if pad_r > 0 or pad_b > 0:
            x = x[:, :, :H, :W].contiguous()
        if self.downsample is not None:
            x = self.downsample(x)
        return x

class SkinDiseaseMamba(nn.Module):
    def __init__(self, dim=48, in_dim=16, depths=[2, 3, 6, 3], window_size=[8, 8, 14, 7],
                 mlp_ratio=2., num_heads=[2, 4, 8, 8], drop_path_rate=0.1,
                 in_chans=3, num_classes=10, drop_rate=0., attn_drop_rate=0., layer_scale=1e-5):
        super().__init__()
        self.num_classes = num_classes
        self.patch_embed = LightweightPatchEmbed(in_chans=in_chans, in_dim=in_dim, dim=dim)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        self.stages = nn.ModuleList()
        for i in range(len(depths)):
            stage = SkinDiseaseMambaStage(
                dim=int(dim * 2 ** i), depth=depths[i], num_heads=num_heads[i],
                window_size=window_size[i], downsample=(i < len(depths) - 1),
                mlp_ratio=mlp_ratio, drop=drop_rate, attn_drop=attn_drop_rate,
                drop_path=dpr[sum(depths[:i]):sum(depths[:i + 1])], layer_scale=layer_scale
            )
            self.stages.append(stage)
        num_features = int(dim * 2 ** (len(depths) - 1))
        self.norm = nn.BatchNorm2d(num_features)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(num_features, num_classes) if num_classes > 0 else nn.Identity()
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.LayerNorm, nn.BatchNorm2d)):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward_features(self, x):
        x = self.patch_embed(x)
        for stage in self.stages:
            x = stage(x)
        x = self.norm(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return x

    def forward(self, x):
        features = self.forward_features(x)
        logits = self.head(features)
        return {'logits': logits, 'features': features}

def skin_disease_mamba(**kwargs):
    defaults = {
        'dim': 48, 'in_dim': 16, 'depths': [2,3,6,3], 
        'window_size': [7, 7, 7, 7], 'mlp_ratio': 2.5,
        'num_heads': [2, 4, 8, 8], 'drop_path_rate': 0.15,
        'layer_scale': 1e-5
    }
    defaults.update(kwargs)
    return SkinDiseaseMamba(**defaults)


## Dataset setup

In [ ]:
def find_dataset():
    """Find dataset in common locations"""
    possible_paths = [
        '/kaggle/input/skin-disease-mamba-dataset-3-classes/CombinedDataset',
        '/kaggle/input/CombinedDataset',
        './CombinedDataset',
        'CombinedDataset'
    ]
    for path in possible_paths:
        if os.path.exists(path):
            return path
    return None

dataset_dir = find_dataset()
if dataset_dir is None:
    print("ERROR: Dataset not found! Please ensure CombinedDataset is available.")
    print("Expected locations:")
    print("  - /kaggle/input/skin-disease-mamba-dataset/CombinedDataset")
    print("  - /kaggle/input/CombinedDataset")
    print("  - ./CombinedDataset")
else:
    print(f"Dataset found at: {os.path.abspath(dataset_dir)}")

# Detect classes and count images
def count_images_in_split(split_path, class_name):
    """Count images in a specific class folder"""
    class_path = os.path.join(split_path, class_name)
    if not os.path.exists(class_path):
        return 0
    # Count image files (common extensions)
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}
    count = 0
    for file in os.listdir(class_path):
        if os.path.splitext(file.lower())[1] in image_extensions:
            count += 1
    return count

train_split_path = os.path.join(dataset_dir, "train") if dataset_dir else None
val_split_path = os.path.join(dataset_dir, "val") if dataset_dir else None
test_split_path = os.path.join(dataset_dir, "test") if dataset_dir else None

if dataset_dir and os.path.exists(train_split_path):
    # Get all class directories from the dataset
    all_class_dirs = [
        d for d in os.listdir(train_split_path)
        if os.path.isdir(os.path.join(train_split_path, d))
    ]
    
    # Define desired class order (modify this list to match your dataset classes)
    # If None, will use sorted order
    desired_class_order = None  # Set to None to use sorted, or define like: ["Healthy", "HFMD", "Not HFMD"]
    
    if desired_class_order is not None:
        # Verify all desired classes exist
        missing_classes = [cls for cls in desired_class_order if cls not in all_class_dirs]
        if missing_classes:
            raise ValueError(f"Missing classes in dataset: {missing_classes}")
        # Use the desired order
        class_names = desired_class_order.copy()
        print(f"\nDetected {len(class_names)} classes (using specified order):")
    else:
        # Use sorted order if no specific order is defined
        class_names = sorted(all_class_dirs)
        print(f"\nDetected {len(class_names)} classes (alphabetically sorted):")
    
    num_classes = len(class_names)
    
    # Count images for each split and class
    train_counts = []
    val_counts = []
    test_counts = []
    
    for idx, name in enumerate(class_names):
        train_count = count_images_in_split(train_split_path, name) if train_split_path else 0
        val_count = count_images_in_split(val_split_path, name) if val_split_path else 0
        test_count = count_images_in_split(test_split_path, name) if test_split_path else 0
        
        train_counts.append(train_count)
        val_counts.append(val_count)
        test_counts.append(test_count)
        
        total = train_count + val_count + test_count
        print(f"  {idx}: {name}  Train: {train_count}, Val: {val_count}, Test: {test_count}, Total: {total}")
    
    # Print summary statistics
    print(f"\nDataset Summary:")
    print(f"  Total train images: {sum(train_counts):,}")
    print(f"  Total val images: {sum(val_counts):,}")
    print(f"  Total test images: {sum(test_counts):,}")
    print(f"  Total images: {sum(train_counts) + sum(val_counts) + sum(test_counts):,}")
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    
    # Bar chart: Images per class for each split
    x = np.arange(len(class_names))
    width = 0.25
    
    bars1 = ax1.bar(x - width, train_counts, width, label='Train', color='#2E86AB', alpha=0.8)
    bars2 = ax1.bar(x, val_counts, width, label='Val', color='#A23B72', alpha=0.8)
    bars3 = ax1.bar(x + width, test_counts, width, label='Test', color='#F18F01', alpha=0.8)
    
    ax1.set_xlabel('Class', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Number of Images', fontsize=12, fontweight='bold')
    ax1.set_title('Image Count per Class by Split', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels([name.replace(' ', '\n') for name in class_names], rotation=0, ha='center', fontsize=9)
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax1.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(height)}',
                        ha='center', va='bottom', fontsize=8)
    
    # Stacked bar chart: Total images per class
    ax2.bar(class_names, train_counts, label='Train', color='#2E86AB', alpha=0.8)
    ax2.bar(class_names, val_counts, bottom=train_counts, label='Val', color='#A23B72', alpha=0.8)
    bottom = [train_counts[i] + val_counts[i] for i in range(len(class_names))]
    ax2.bar(class_names, test_counts, bottom=bottom, label='Test', color='#F18F01', alpha=0.8)
    
    ax2.set_xlabel('Class', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Total Number of Images', fontsize=12, fontweight='bold')
    ax2.set_title('Total Images per Class', fontsize=14, fontweight='bold')
    ax2.set_xticks(np.arange(len(class_names)))
    ax2.set_xticklabels([name.replace(' ', '\n') for name in class_names], rotation=0, ha='center', fontsize=9)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add total labels on top of stacked bars
    totals = [train_counts[i] + val_counts[i] + test_counts[i] for i in range(len(class_names))]
    for i, (name, total) in enumerate(zip(class_names, totals)):
        ax2.text(i, total, f'{total}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
else:
    print(f"ERROR: Train split not found at {train_split_path}" if train_split_path else "")
    num_classes = 8  # fallback
    class_names = [f'Class_{i}' for i in range(num_classes)]
    train_counts = []
    val_counts = []
    test_counts = []


## Trainning functions

In [ ]:
class ClassificationLoss(nn.Module):
    def __init__(self, smoothing=0.0):
        super().__init__()
        self.cls_loss = LabelSmoothingCrossEntropy(smoothing=smoothing)
        self.soft_target_loss = SoftTargetCrossEntropy()
    def forward(self, outputs, targets):
        logits = outputs['logits']
        if targets.dtype == torch.float32 and targets.dim() > 1:
            loss = self.soft_target_loss(logits, targets)
        else:
            targets = targets.long()
            loss = self.cls_loss(logits, targets)
        return loss, {'loss': loss.item()}

def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModel Parameters:")
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    print(f"  Model size: {total_params * 4 / 1024 / 1024:.2f} MB (float32)")

def train_one_epoch(epoch, model, loader, optimizer, loss_fn, log_interval=50,
                   lr_scheduler=None, amp_autocast=suppress, loss_scaler=None):
    batch_time_m = utils.AverageMeter()
    data_time_m = utils.AverageMeter()
    losses_m = utils.AverageMeter()
    top1_m = utils.AverageMeter()
    model.train()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    end = time.time()
    last_idx = len(loader) - 1
    num_updates = epoch * len(loader)
    for batch_idx, (input, target) in enumerate(loader):
        data_time_m.update(time.time() - end)
        input, target = input.cuda(non_blocking=True), target.cuda(non_blocking=True)
        with amp_autocast('cuda'):
            output = model(input)
            loss, loss_dict = loss_fn(output, target)
        losses_m.update(loss_dict['loss'], input.size(0))
        acc1, _ = utils.accuracy(output['logits'], target, topk=(1, 5))
        top1_m.update(acc1.item(), input.size(0))
        optimizer.zero_grad()
        if loss_scaler is not None:
            loss_scaler(loss, optimizer, clip_grad=1.0, clip_mode='norm',
                       parameters=model_parameters(model), create_graph=False)
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        del input, target, output, loss
        if lr_scheduler is not None:
            lr_scheduler.step_update(num_updates)
        num_updates += 1
        batch_time_m.update(time.time() - end)
        if batch_idx % 10 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
        if batch_idx % log_interval == 0:
            lrl = [param_group['lr'] for param_group in optimizer.param_groups]
            lr = sum(lrl) / len(lrl)
            if torch.cuda.is_available() and batch_idx == 0:
                memory_allocated = torch.cuda.memory_allocated() / 1024**3
                memory_reserved = torch.cuda.memory_reserved() / 1024**3
                print(f'GPU Memory - Allocated: {memory_allocated:.2f} GB, Reserved: {memory_reserved:.2f} GB')
            print(f'Train: {epoch} [{batch_idx:>4d}/{len(loader)} ({100. * batch_idx / last_idx:>3.0f}%)] '
                  f'Loss: {losses_m.val:#.4g} ({losses_m.avg:#.3g}) '
                  f'Acc@1: {top1_m.val:#.3g} ({top1_m.avg:#.3g}) '
                  f'Time: {batch_time_m.val:.3f}s LR: {lr:.3e}')
        end = time.time()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    return OrderedDict([('loss', losses_m.avg), ('top1', top1_m.avg)])

def validate(model, loader, loss_fn, log_interval=50, amp_autocast=suppress):
    batch_time_m = utils.AverageMeter()
    losses_m = utils.AverageMeter()
    top1_m = utils.AverageMeter()
    top5_m = utils.AverageMeter()
    model.eval()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    end = time.time()
    last_idx = len(loader) - 1
    with torch.no_grad():
        for batch_idx, (input, target) in enumerate(loader):
            input, target = input.cuda(non_blocking=True), target.cuda(non_blocking=True)
            with amp_autocast('cuda'):
                output = model(input)
                logits = output['logits']
                loss = loss_fn(logits, target.long())
            acc1, acc5 = utils.accuracy(logits, target, topk=(1, 5))
            losses_m.update(loss.item(), input.size(0))
            top1_m.update(acc1.item(), input.size(0))
            top5_m.update(acc5.item(), input.size(0))
            del input, target, output, logits, loss
            batch_time_m.update(time.time() - end)
            end = time.time()
            if batch_idx % 20 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
            if batch_idx % log_interval == 0:
                print(f'Test: [{batch_idx:>4d}/{last_idx}] '
                      f'Time: {batch_time_m.val:.3f} ({batch_time_m.avg:.3f}) '
                      f'Loss: {losses_m.val:>7.4f} ({losses_m.avg:>6.4f}) '
                      f'Acc@1: {top1_m.val:>7.4f} ({top1_m.avg:>7.4f})')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return OrderedDict([('loss', losses_m.avg), ('top1', top1_m.avg), ('top5', top5_m.avg)])

def plot_training_curves(train_losses, val_losses, train_accs, val_accs, save_path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    ax1.plot(train_losses, label='Training Loss', color='blue', linewidth=2)
    ax1.plot(val_losses, label='Validation Loss', color='red', linewidth=2)
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    final_train_loss = train_losses[-1] if train_losses else 0
    final_val_loss = val_losses[-1] if val_losses else 0
    ax1.text(0.02, 0.98, f'Final Train Loss: {final_train_loss:.4f}\nFinal Val Loss: {final_val_loss:.4f}', 
             transform=ax1.transAxes, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax2.plot(train_accs, label='Training Accuracy', color='blue', linewidth=2)
    ax2.plot(val_accs, label='Validation Accuracy', color='red', linewidth=2)
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Training and Validation Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    final_train_acc = train_accs[-1] if train_accs else 0
    final_val_acc = val_accs[-1] if val_accs else 0
    ax2.text(0.02, 0.02, f'Final Train Acc: {final_train_acc:.2f}%\nFinal Val Acc: {final_val_acc:.2f}%', 
             transform=ax2.transAxes, verticalalignment='bottom', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Training curves saved to: {save_path}")

def plot_confusion_matrix(all_targets, all_predictions, class_names, save_path):
    cm = confusion_matrix(all_targets, all_predictions)
    row_sums = cm.sum(axis=1)
    cm_percent = np.zeros_like(cm, dtype=float)
    for i in range(len(class_names)):
        if row_sums[i] > 0:
            cm_percent[i, :] = cm[i, :].astype('float') / row_sums[i] * 100
        else:
            cm_percent[i, :] = 0.0
    annotations = []
    for i in range(len(class_names)):
        row = []
        for j in range(len(class_names)):
            count = int(cm[i, j])
            percent = cm_percent[i, j]
            if count > 0:
                row.append(f'{count}\n({percent:.1f}%)')
            else:
                row.append('0\n(0.0%)')
        annotations.append(row)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=annotations, fmt='', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'}, linewidths=0.5, linecolor='gray')
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to: {save_path}")


## Training hyperparameters

In [ ]:
# Main training function - all code inlined
def train_model(data_dir, num_classes, epochs=100, batch_size=128, lr=0.0005, 
                warmup_epochs=5, patience_epochs=20, seed=42, output_dir='./results'):
    """Main training function"""
    utils.setup_default_logging()
    utils.random_seed(seed, 0)
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print(f'GPU Memory cleared. Available: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')
    
    # Create model
    model = skin_disease_mamba(num_classes=num_classes, drop_path_rate=0.01)
    
    # Check for multiple GPUs and use DataParallel if available
    if torch.cuda.is_available():
        num_gpus = torch.cuda.device_count()
        if num_gpus > 1:
            print(f'Using {num_gpus} GPUs with DataParallel')
            model = model.cuda()
            model = nn.DataParallel(model)
        else:
            model.cuda()
            print(f'Using single GPU: {torch.cuda.get_device_name(0)}')
    else:
        print('CUDA not available, using CPU (this will be very slow)')
    
    count_parameters(model)
    
    # Setup mixed precision
    amp_autocast = torch.amp.autocast
    loss_scaler = NativeScaler()
    print('Using native Torch AMP for mixed precision training.')
    
    # Create optimizer
    optimizer = create_optimizer_v2(model, opt='adamw', lr=lr, weight_decay=0.01, momentum=0.9, eps=1e-8)
    
    # Create datasets and loaders
    class Args:
        def __init__(self):
            self.input_size = (3, 224, 224)
            self.interpolation = 'bicubic'
            self.mean = (0.485, 0.456, 0.406)
            self.std = (0.229, 0.224, 0.225)
            self.crop_pct = 0.9
    
    data_config = Args()
    dataset_train = create_dataset('', root=data_dir, split='train', is_training=True, batch_size=batch_size)
    dataset_eval = create_dataset('', root=data_dir, split='val', is_training=False, batch_size=batch_size)
    dataset_test = create_dataset('', root=data_dir, split='test', is_training=False, batch_size=batch_size)
    
    loader_train = create_loader(dataset_train, input_size=data_config.input_size, batch_size=batch_size,
                                 is_training=True, use_prefetcher=True, no_aug=False, re_prob=0.05,
                                 re_mode='pixel', re_count=1, scale=[0.8, 1.0], ratio=[3./4., 4./3.],
                                 hflip=0.5, vflip=0.2, color_jitter=0.05, interpolation=data_config.interpolation,
                                 mean=data_config.mean, std=data_config.std, num_workers=3, distributed=False, pin_memory=True)
    
    loader_eval = create_loader(dataset_eval, input_size=data_config.input_size, batch_size=batch_size,
                                is_training=False, use_prefetcher=True, interpolation=data_config.interpolation,
                                mean=data_config.mean, std=data_config.std, num_workers=3, distributed=False,
                                crop_pct=data_config.crop_pct, pin_memory=True)
    
    loader_test = create_loader(dataset_test, input_size=data_config.input_size, batch_size=batch_size,
                               is_training=False, use_prefetcher=True, interpolation=data_config.interpolation,
                               mean=data_config.mean, std=data_config.std, num_workers=3, distributed=False,
                               crop_pct=data_config.crop_pct, pin_memory=True)
    
    # Setup loss and scheduler
    train_loss_fn = ClassificationLoss(smoothing=0.01).cuda()
    validate_loss_fn = nn.CrossEntropyLoss().cuda()
    
    class SchedulerArgs:
        def __init__(self):
            self.sched = 'cosine'
            self.epochs = epochs
            self.warmup_epochs = warmup_epochs
            self.min_lr = 5e-6
    
    scheduler_args = SchedulerArgs()
    lr_scheduler, num_epochs = create_scheduler(scheduler_args, optimizer)
    
    # Setup output directory
    exp_name = f"skin_disease_mamba-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    output_path = os.path.join(output_dir, exp_name)
    os.makedirs(output_path, exist_ok=True)
    
    # Training history
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    
    print(f'Starting training for {num_epochs} epochs...')
    best_metric = 0.0
    best_epoch = 0
    patience_counter = 0
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        train_metrics = train_one_epoch(epoch, model, loader_train, optimizer, train_loss_fn,
                                       log_interval=50, lr_scheduler=lr_scheduler, amp_autocast=amp_autocast, loss_scaler=loss_scaler)
        eval_metrics = validate(model, loader_eval, validate_loss_fn, log_interval=50, amp_autocast=amp_autocast)
        
        train_losses.append(train_metrics['loss'])
        val_losses.append(eval_metrics['loss'])
        train_accs.append(train_metrics.get('top1', 0))
        val_accs.append(eval_metrics['top1'])
        
        if lr_scheduler is not None:
            lr_scheduler.step(epoch + 1, eval_metrics.get('top1', 0))
        
        improved = False
        if eval_metrics['loss'] < best_val_loss:
            best_val_loss = eval_metrics['loss']
            improved = True
        
        if improved:
            best_metric = eval_metrics['top1']
            best_epoch = epoch
            patience_counter = 0
            # Handle DataParallel: remove 'module.' prefix for cleaner checkpoint
            model_state = model.state_dict()
            if any(key.startswith('module.') for key in model_state.keys()):
                model_state = {k.replace('module.', ''): v for k, v in model_state.items()}
            
            torch.save({'epoch': epoch, 'state_dict': model_state, 'optimizer': optimizer.state_dict(),
                       'best_acc': best_metric}, os.path.join(output_path, 'model_best.pth.tar'))
        else:
            patience_counter += 1
            if patience_counter >= patience_epochs:
                print(f"Early stopping triggered at epoch {epoch+1}.")
                break
        
        print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_metrics["loss"]:.4f} - '
              f'Val Acc: {eval_metrics["top1"]:.4f} - Best Acc: {best_metric:.4f} (epoch {best_epoch+1}) - '
              f'Val Loss: {eval_metrics["loss"]:.4f} (best: {best_val_loss:.4f}) - '
              f'Patience: {patience_counter}/{patience_epochs}')
    
    # Plot training curves
    plot_training_curves(train_losses, val_losses, train_accs, val_accs, os.path.join(output_path, 'training_curves.png'))
    
    # Test evaluation
    print('Evaluating on test set...')
    test_metrics = validate(model, loader_test, validate_loss_fn, log_interval=50, amp_autocast=amp_autocast)
    
    # Generate confusion matrix
    model.eval()
    all_targets = []
    all_predictions = []
    with torch.no_grad():
        for input, target in loader_eval:
            input = input.cuda()
            output = model(input)
            predictions = output['logits'].argmax(dim=1)
            all_targets.extend(target.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
    
    plot_confusion_matrix(all_targets, all_predictions, class_names, os.path.join(output_path, 'confusion_matrix_val.png'))
    
    print(f'\nTraining completed! Best validation accuracy: {best_metric:.4f} (epoch {best_epoch+1})')
    print(f'Test accuracy: {test_metrics["top1"]:.4f}')
    
    return model, output_path


## Train the model

In [ ]:
# Run training - modify parameters as needed
if dataset_dir is not None:
    # Use /kaggle/working for Kaggle, ./results for local
    output_base = '/kaggle/working/results' if os.path.exists('/kaggle/working') else './results'
    os.makedirs(output_base, exist_ok=True)
    
    # Train with different seeds (modify as needed)
    seeds = [42]  # Add more seeds: [42, 123, 555, 2025] for multiple runs
    
    for seed in seeds:
        print(f"\n{'='*60}")
        print(f"Training with seed: {seed}")
        print(f"{'='*60}\n")
        
        model, output_path = train_model(
            data_dir=dataset_dir,
            num_classes=num_classes,
            epochs=100,
            batch_size=128,
            lr=0.0005,
            warmup_epochs=5,
            patience_epochs=20,
            seed=seed,
            output_dir=output_base
        )
        
        print(f"\nExperiment completed. Results saved to: {output_path}")
else:
    print("Cannot start training: dataset not found!")
